# Orthography Large Experiment

This notebook tracks the expanded orthography experiment as `*_output.jsonl` files land in `batches/orthography_large_exp/`. It mirrors the newer analysis style used in the agreement and error-analysis notebooks: deterministic loaders, progress snapshots that tolerate partial downloads, and compact plotting built around a fixed five-condition orthography palette.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import re
import sys
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyrootutils
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 3)

PROJECT_ROOT = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

EXP_NAME = "orthography_large"
EXP_DATA_DIR = DATA_DIR / "orthography_large_exp"
EXP_BATCH_DIR = BATCHES_DIR / "orthography_large_exp"
GRAMMAR_LIST = EXP_DATA_DIR / "orthography_large_grammars.txt"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert EXP_DATA_DIR.exists(), EXP_DATA_DIR
assert EXP_BATCH_DIR.exists(), EXP_BATCH_DIR
assert GRAMMAR_LIST.exists(), GRAMMAR_LIST

## Define aesthetics

In [ ]:
import aesthetics as aes  # noqa: F401

sns = aes.sns
plt = aes.plt
mtick = aes.mtick

MODEL_ORDER = [
    model
    for model in aes.MODEL_ORDER
    if model.startswith("gpt-5") or "gemma-3" in model
]
ORTHOGRAPHY_ORDER = [
    "Latin → Latin",
    "Latin → Latin (diacritics)",
    "Latin → Cyrillic",
    "Latin → Hebrew",
    "Latin → Hebrew (pointed)",
]
ORTHOGRAPHY_LABELS = {
    "latin": "Latin → Latin",
    "latin_diacritic": "Latin → Latin (diacritics)",
    "cyrillic": "Latin → Cyrillic",
    "hebrew": "Latin → Hebrew (pointed)",
    "hebrew_unpointed": "Latin → Hebrew",
}
PALETTE_ORTHOGRAPHY_LARGE = aes.darken(
    {
        "Latin → Latin": sns.color_palette("Reds", n_colors=7)[1],
        "Latin → Latin (diacritics)": sns.color_palette("Reds", n_colors=7)[3],
        "Latin → Cyrillic": sns.color_palette("Greens", n_colors=7)[2],
        "Latin → Hebrew (pointed)": sns.color_palette("Blues", n_colors=7)[3],
        "Latin → Hebrew": sns.color_palette("Blues", n_colors=7)[5],
    },
    by=0.15,
)
METRICS_FOR_PLOT = [
    ("exact_match", "Exact Match"),
    ("bow_match", "Bag of Words"),
    ("bleu", "BLEU Score"),
    ("chrF++", "chrF++"),
]

## Load grammars and samples

In [ ]:
ANSWER_RE = re.compile(
    r"final\s*answer\s*(?::|-|—)?\s*(?:is\s*)?([^\n]+)",
    re.IGNORECASE | re.DOTALL,
)
CYRILLIC_RE = re.compile(r"[а-яА-Я]")
HEBREW_RE = re.compile(r"[\u0590-\u05FF]")


def fuzzy_model(model: str | None) -> str:
    return re.sub(r"-\d{4}-\d{2}-\d{2}$", "", model or "")


def extract_answer(text: str | None) -> str | None:
    if not text:
        return None
    matches = ANSWER_RE.findall(text)
    if not matches:
        return None
    answer = matches[-1].strip()
    answer = re.sub(r"[^\w\s]", "", answer, flags=re.UNICODE).strip()
    return answer or None


def usage_tuple(body: dict) -> tuple[int | None, int | None, int | None]:
    usage = body.get("usage", {}) or {}
    return (
        usage.get("prompt_tokens", usage.get("promptTokens")),
        usage.get("completion_tokens", usage.get("completionTokens")),
        usage.get("total_tokens", usage.get("totalTokens")),
    )


def tokenize(text: str | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return text.split()


def bag_equal(left: str | None, right: str | None) -> bool:
    return Counter(tokenize(left)) == Counter(tokenize(right))


def detect_script(text: str | None) -> str:
    if not isinstance(text, str) or not text.strip():
        return "empty"
    if CYRILLIC_RE.search(text):
        return "cyrillic"
    if HEBREW_RE.search(text):
        return "hebrew"
    for char in text:
        if char.isalpha() and "LATIN" in unicodedata.name(char, ""):
            return "latin"
    return "other"


def load_grammars(grammar_list: Path) -> pd.DataFrame:
    grammar_ids = [
        line.strip() for line in grammar_list.read_text().splitlines() if line.strip()
    ]
    rows = []
    for grammar_id in grammar_ids:
        grammar = json.loads((EXP_DATA_DIR / f"grammar_{grammar_id}.json").read_text())
        rows.append(
            {
                "grammar_name": grammar_id,
                "n_rules": pd.to_numeric(grammar.get("n_rules"), errors="coerce"),
                "n_words": pd.to_numeric(grammar.get("n_words"), errors="coerce"),
                "grammar_size": pd.to_numeric(grammar.get("n_rules"), errors="coerce")
                + pd.to_numeric(grammar.get("n_words"), errors="coerce"),
                "target_orthography_key": grammar["b"].get("orthography", "unknown"),
                "target_orthography": ORTHOGRAPHY_LABELS.get(
                    grammar["b"].get("orthography", "unknown"),
                    grammar["b"].get("orthography", "unknown"),
                ),
                "syllable_structure": grammar["a"].get("syllable_structure"),
                "target_vocab": sum(
                    [
                        grammar["b"][key]
                        for key in [
                            "verbs",
                            "nouns",
                            "propns",
                            "prons",
                            "adjs",
                            "det_def",
                            "det_indef",
                            "comps",
                        ]
                    ],
                    [],
                ),
            }
        )
    grammar_df = pd.DataFrame(rows)
    grammar_df["target_orthography"] = pd.Categorical(
        grammar_df["target_orthography"],
        categories=ORTHOGRAPHY_ORDER,
        ordered=True,
    )
    return grammar_df


def load_samples(grammar_ids: list[str]) -> pd.DataFrame:
    rows = []
    for grammar_id in grammar_ids:
        sample_path = EXP_DATA_DIR / f"samples_{grammar_id}.jsonl"
        with open(sample_path) as handle:
            for sample_id, line in enumerate(handle):
                sample = json.loads(line)
                rows.append(
                    {
                        "grammar_name": grammar_id,
                        "sample_id": str(sample_id),
                        "input_sentence": sample.get("left_phonetic")
                        or sample.get("left"),
                        "output_sentence": sample.get("right_phonetic")
                        or sample.get("right"),
                        "depth": pd.to_numeric(sample.get("depth"), errors="coerce"),
                    }
                )
    return pd.DataFrame(rows)


grammars_df = load_grammars(GRAMMAR_LIST)
samples_df = load_samples(grammars_df["grammar_name"].tolist()).merge(
    grammars_df[
        ["grammar_name", "target_orthography", "grammar_size", "n_rules", "n_words"]
    ],
    on="grammar_name",
    how="left",
)
samples_df["input_length"] = samples_df["input_sentence"].map(
    lambda text: len(tokenize(text))
)

display(
    samples_df.groupby(["target_orthography", "grammar_size"], observed=False)
    .size()
    .rename("n_samples")
    .reset_index()
)

condition_examples_df = (
    samples_df.sort_values(["target_orthography", "grammar_size", "depth", "sample_id"])
    .groupby("target_orthography", observed=False)
    .head(1)[
        ["target_orthography", "grammar_size", "input_sentence", "output_sentence"]
    ]
    .reset_index(drop=True)
)
condition_examples_df

## Load available batch outputs

In [ ]:
CUSTOM_ID_RE = re.compile(
    r"^(?P<grammar_name>[0-9a-f]+)-(?P<input_hash>[0-9a-f]+)-sample-(?P<sample_id>\d+)$"
)


def load_outputs(batch_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(batch_dir.glob("*_output.jsonl")):
        with open(path) as handle:
            for line in handle:
                item = json.loads(line)
                body = (item.get("response") or {}).get("body") or {}
                choices = body.get("choices") or []
                message = (
                    ((choices[0] or {}).get("message") or {}).get("content")
                    if choices
                    else None
                )
                match = CUSTOM_ID_RE.match(item.get("custom_id", ""))
                if not match:
                    continue
                prompt_tokens, completion_tokens, total_tokens = usage_tuple(body)
                rows.append(
                    {
                        "batch_file": path.name,
                        "batch_id": path.name.replace("_output.jsonl", ""),
                        "custom_id": item.get("custom_id"),
                        "grammar_name": match.group("grammar_name"),
                        "input_hash": match.group("input_hash"),
                        "sample_id": match.group("sample_id"),
                        "model": body.get("model"),
                        "fuzzy_model": fuzzy_model(body.get("model")),
                        "model_response": message,
                        "model_answer": extract_answer(message),
                        "status_code": (item.get("response") or {}).get("status_code"),
                        "error": item.get("error"),
                        "prompt_tokens": prompt_tokens,
                        "completion_tokens": completion_tokens,
                        "total_tokens": total_tokens,
                    }
                )
    return pd.DataFrame(rows)


def load_inputs(batch_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(batch_dir.glob("inputs_*.jsonl")):
        if path.name.endswith("_output.jsonl"):
            continue
        with open(path) as handle:
            for line in handle:
                item = json.loads(line)
                body = item.get("body") or {}
                metadata = body.get("metadata") or {}
                match = CUSTOM_ID_RE.match(item.get("custom_id", ""))
                rows.append(
                    {
                        "custom_id": item.get("custom_id"),
                        "input_file": path.name,
                        "fuzzy_model": fuzzy_model(body.get("model")),
                        "grammar_name": metadata.get("grammar_name")
                        or (match.group("grammar_name") if match else None),
                        "sample_id": (
                            str(metadata.get("sample_id"))
                            if metadata.get("sample_id") is not None
                            else (match.group("sample_id") if match else None)
                        ),
                        "depth": pd.to_numeric(metadata.get("depth"), errors="coerce"),
                    }
                )
    return pd.DataFrame(rows)


outputs_df = load_outputs(EXP_BATCH_DIR)
inputs_df = load_inputs(EXP_BATCH_DIR)
print(f"Output rows loaded: {len(outputs_df)}")
print(f"Input rows indexed: {len(inputs_df)}")

if len(outputs_df):
    display(
        outputs_df.groupby(["batch_file", "fuzzy_model"], observed=False)
        .size()
        .rename("n_rows")
        .reset_index()
    )
else:
    print("No batch output files found yet.")

## Merge outputs with samples and grammars

In [ ]:
merged_df = outputs_df.merge(
    inputs_df.drop_duplicates(subset=["custom_id", "fuzzy_model"]),
    on=["custom_id", "fuzzy_model"],
    how="left",
    suffixes=("", "_input"),
)

for column in ["grammar_name", "sample_id", "depth"]:
    fallback_column = f"{column}_input"
    if fallback_column in merged_df.columns:
        merged_df[column] = merged_df[column].combine_first(merged_df[fallback_column])
        merged_df = merged_df.drop(columns=[fallback_column])

if len(merged_df):
    merged_df = merged_df.drop_duplicates(subset=["batch_id", "custom_id"]).copy()
    merged_df = merged_df.merge(
        samples_df[
            [
                "grammar_name",
                "sample_id",
                "input_sentence",
                "output_sentence",
                "input_length",
            ]
        ],
        on=["grammar_name", "sample_id"],
        how="left",
        validate="many_to_one",
    )
    merged_df = merged_df.merge(
        grammars_df[
            [
                "grammar_name",
                "target_orthography",
                "target_orthography_key",
                "grammar_size",
                "n_rules",
                "n_words",
                "target_vocab",
                "syllable_structure",
            ]
        ],
        on="grammar_name",
        how="left",
        validate="many_to_one",
    )

    merged_df["answered"] = merged_df["model_answer"].notna()
    merged_df["target_script"] = merged_df["output_sentence"].map(detect_script)
    merged_df["answer_script"] = merged_df["model_answer"].map(detect_script)
    merged_df["script_match"] = merged_df["target_script"] == merged_df["answer_script"]

    try:
        merged_df["input_length_quintile"] = pd.qcut(
            merged_df["input_length"], q=5, duplicates="drop"
        )
        merged_df["input_length_quintile_mid"] = merged_df["input_length_quintile"].map(
            lambda interval: (interval.left + interval.right) / 2
            if pd.notna(interval)
            else np.nan
        )
    except ValueError:
        merged_df["input_length_quintile"] = pd.NA
        merged_df["input_length_quintile_mid"] = np.nan

    completion_snapshot = (
        merged_df.groupby(["fuzzy_model", "target_orthography"], observed=False)
        .agg(rows=("custom_id", "size"), answered=("answered", "sum"))
        .reset_index()
    )
    completion_snapshot["answer_rate"] = completion_snapshot[
        "answered"
    ] / completion_snapshot["rows"].replace(0, np.nan)
    display(completion_snapshot.sort_values(["fuzzy_model", "target_orthography"]))

    expected_per_model = len(samples_df)
    print(
        (
            merged_df.groupby("fuzzy_model", observed=False)["custom_id"]
            .nunique()
            .rename("completed_rows")
            .reset_index()
            .assign(
                expected_rows=expected_per_model,
                completion_rate=lambda df: df["completed_rows"] / df["expected_rows"],
            )
        )
    )
else:
    print("No outputs merged yet.")

merged_df.head() if len(merged_df) else merged_df

## Accuracy metrics

In [ ]:
try:
    import evaluate
except ImportError:
    evaluate = None

try:
    import sacrebleu
except ImportError:
    sacrebleu = None

if len(merged_df):

    def exact_match(row) -> bool:
        return (
            bool(row["model_answer"]) and row["model_answer"] == row["output_sentence"]
        )

    def bow_match(row) -> bool:
        return bag_equal(row["model_answer"], row["output_sentence"])

    def edit_similarity(row) -> float:
        return SequenceMatcher(
            None, row["model_answer"] or "", row["output_sentence"] or ""
        ).ratio()

    def bleu_score(row) -> float:
        if sacrebleu is None:
            return np.nan
        return (
            sacrebleu.sentence_bleu(
                row["model_answer"] or "", [row["output_sentence"] or ""]
            ).score
            / 100.0
        )

    def num_oov_words(row) -> int:
        if not row["model_answer"]:
            return 0
        return len(set(tokenize(row["model_answer"])) - set(row["target_vocab"] or []))

    merged_df["exact_match"] = merged_df.apply(exact_match, axis=1)
    merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
    merged_df["edit_similarity"] = merged_df.apply(edit_similarity, axis=1)
    merged_df["bleu"] = merged_df.apply(bleu_score, axis=1)
    merged_df["num_oov_words"] = merged_df.apply(num_oov_words, axis=1)

    if evaluate is not None:
        chrf = evaluate.load("chrf")
        merged_df["chrF++"] = [
            chrf.compute(
                predictions=[pred or ""],
                references=[[ref or ""]],
                beta=2,
                word_order=2,
            )["score"]
            / 100.0
            for pred, ref in zip(
                merged_df["model_answer"], merged_df["output_sentence"]
            )
        ]
    else:
        merged_df["chrF++"] = np.nan

    summary_by_model = (
        merged_df.groupby("fuzzy_model", observed=False)
        .agg(
            rows=("custom_id", "size"),
            answered=("answered", "sum"),
            exact_match=("exact_match", "mean"),
            bow_match=("bow_match", "mean"),
            script_match=("script_match", "mean"),
            edit_similarity=("edit_similarity", "mean"),
            bleu=("bleu", "mean"),
            chrf_pp=("chrF++", "mean"),
            mean_prompt_tokens=("prompt_tokens", "mean"),
            mean_completion_tokens=("completion_tokens", "mean"),
        )
        .reset_index()
    )
    for column in [
        "exact_match",
        "bow_match",
        "script_match",
        "edit_similarity",
        "bleu",
        "chrf_pp",
    ]:
        summary_by_model[column] = (100 * summary_by_model[column]).round(2)
    display(summary_by_model)

    by_condition = (
        merged_df.groupby(["target_orthography", "fuzzy_model"], observed=False)
        .agg(
            rows=("custom_id", "size"),
            answered=("answered", "sum"),
            exact_match=("exact_match", "mean"),
            bow_match=("bow_match", "mean"),
            script_match=("script_match", "mean"),
            edit_similarity=("edit_similarity", "mean"),
            bleu=("bleu", "mean"),
            chrf_pp=("chrF++", "mean"),
            mean_oov_words=("num_oov_words", "mean"),
        )
        .reset_index()
    )
    for column in [
        "exact_match",
        "bow_match",
        "script_match",
        "edit_similarity",
        "bleu",
        "chrf_pp",
    ]:
        by_condition[column] = (100 * by_condition[column]).round(2)
    display(by_condition.sort_values(["fuzzy_model", "target_orthography"]))
else:
    print("No outputs available for scoring yet.")

## Plots

In [ ]:
# PLOT_MODEL = "gpt-5-nano"
PLOT_MODEL = None

In [ ]:
if len(merged_df):
    available_models = [
        model
        for model in MODEL_ORDER
        if model in set(merged_df["fuzzy_model"].dropna())
    ]
    selected_model = (
        PLOT_MODEL
        if PLOT_MODEL is not None
        else (available_models[-1] if available_models else None)
    )
    print(f"Plotting model: {selected_model}")

    plot_source_df = merged_df.loc[merged_df["fuzzy_model"] == selected_model].copy()
    plot_source_df["target_orthography"] = pd.Categorical(
        plot_source_df["target_orthography"],
        categories=ORTHOGRAPHY_ORDER,
        ordered=True,
    )

    plot_metric_df = plot_source_df.melt(
        id_vars=[
            "target_orthography",
            "grammar_size",
            "depth",
            "input_length",
            "input_length_quintile_mid",
            "prompt_tokens",
            "completion_tokens",
        ],
        value_vars=[metric for metric, _ in METRICS_FOR_PLOT],
        var_name="metric_key",
        value_name="match_value",
    )
    plot_metric_df["match_type"] = plot_metric_df["metric_key"].replace(
        dict(METRICS_FOR_PLOT)
    )

    fig = plt.figure(
        figsize=(aes.COLM_PAPER_WIDTH_IN, 1 * aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
    )
    grid = fig.add_gridspec(2, len(METRICS_FOR_PLOT), wspace=0.12, hspace=0.7)

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        for row_idx, x_column, xlabel in [
            (0, "grammar_size", "Grammar size"),
            (1, "input_length_quintile_mid", "Sentence length (binned to 5 quantiles)"),
        ]:
            for col_idx, (_, metric_label) in enumerate(METRICS_FOR_PLOT):
                ax = fig.add_subplot(grid[row_idx, col_idx])
                sns.lineplot(
                    data=plot_metric_df.loc[
                        plot_metric_df["match_type"] == metric_label
                    ],
                    x=x_column,
                    y="match_value",
                    hue="target_orthography",
                    style="target_orthography",
                    hue_order=ORTHOGRAPHY_ORDER,
                    style_order=ORTHOGRAPHY_ORDER,
                    palette=PALETTE_ORTHOGRAPHY_LARGE,
                    markers=True,
                    dashes=False,
                    errorbar="ci",
                    ax=ax,
                )
                ax.set_ylim(-0.05, 1.05)
                ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
                if x_column == "grammar_size":
                    ax.xaxis.set_major_formatter(aes.KMB_FORMATTER)
                    ax.set_xlim(0, 10_000)
                    labels = ax.get_xticklabels()
                    labels[-1].set_ha("right")
                ax.set_title(
                    metric_label if row_idx == 0 else "",
                    fontweight="normal",
                    loc="center",
                )
                ax.set_xlabel(xlabel if col_idx == 0 else "")
                ax.set_ylabel("Mean score" if col_idx == 0 else "")
                if col_idx != 0:
                    ax.yaxis.set_ticklabels([])
                legend = ax.get_legend()
                if (
                    row_idx == 0
                    and col_idx == len(METRICS_FOR_PLOT) - 1
                    and legend is not None
                ):
                    legend.set_title("")
                    sns.move_legend(
                        ax, "upper left", bbox_to_anchor=(1.02, 1.0), frameon=False
                    )
                elif legend is not None:
                    legend.remove()

    aes.save_figure(f"{selected_model}_orthography_large_overview", fig=fig)
    plt.show()
else:
    print("No scored outputs available yet for plotting.")

In [ ]:
if len(merged_df):
    distribution_df = merged_df.copy()
    distribution_df["predicted_length"] = distribution_df["model_answer"].map(
        lambda text: len(tokenize(text))
    )
    distribution_df["target_length"] = distribution_df["output_sentence"].map(
        lambda text: len(tokenize(text))
    )
    hue_order = aes.ordered_models(distribution_df["fuzzy_model"].dropna().unique())

    fig, axes = plt.subplots(
        1, 2, figsize=(aes.COLM_PAPER_WIDTH_IN, 1.1 * aes.FIG_HEIGHT_SINGLE_ROW_IN)
    )

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        sns.histplot(
            data=distribution_df,
            x="predicted_length",
            hue="fuzzy_model",
            hue_order=hue_order,
            element="step",
            stat="density",
            common_norm=False,
            palette=aes.PALETTE_MODELS,
            ax=axes[0],
        )
        sns.kdeplot(
            data=distribution_df,
            x="target_length",
            color="black",
            ax=axes[0],
            label="Target",
        )
        if axes[0].legend_ is not None:
            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles, labels, frameon=False, title="")
        axes[0].set_xlabel("Predicted translation length")
        axes[0].set_ylabel("Density")

        sns.histplot(
            data=distribution_df,
            x="completion_tokens",
            hue="fuzzy_model",
            hue_order=hue_order,
            element="step",
            stat="density",
            common_norm=False,
            palette=aes.PALETTE_MODELS,
            ax=axes[1],
        )
        axes[1].xaxis.set_major_formatter(aes.KMB_FORMATTER)
        axes[1].set_xlabel("Completion tokens")
        axes[1].set_ylabel("Density")
        if axes[1].legend_ is not None:
            sns.move_legend(axes[1], "upper right", frameon=False, title="")

    plt.tight_layout()
    aes.save_figure("orthography_large_length_and_ttc", fig=fig)
    plt.show()
else:
    print("No merged outputs available yet for distribution plots.")

## Notes

- Re-run the notebook as additional `*_output.jsonl` files arrive in [`/Users/jacksonpetty/Development/llm-scfg/batches/orthography_large_exp`](/Users/jacksonpetty/Development/llm-scfg/batches/orthography_large_exp).
- `PLOT_MODEL = None` auto-selects the highest-priority available model from `MODEL_ORDER`; set it explicitly if you want a different model for the saved figures.
- If `evaluate` or `sacrebleu` are unavailable in the kernel, the notebook still runs and leaves those metric columns as `NaN`.